In [1]:
##### Converts raw rasters on land use decesion making probabilities into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # probability of being driven by economic objectives
    # probability of being driven by survival objectives

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# objective rasters
economic = f"{cd}/Data/Raw/Predictors/Land_use_decision_making_Malek/economic.tif"
survival = f"{cd}/Data/Raw/Predictors/Land_use_decision_making_Malek/survival.tif"

# save paths
pixels_economic = f"{cd}/Data/Clean/Predictors/Rasters/economic_objective_prob.tif"
pixels_survival = f"{cd}/Data/Clean/Predictors/Rasters/survival_objective_prob.tif"

vectors = f"{cd}/Data/Clean/Predictors/Vectors/decison_making_objective_prob.csv"

In [3]:
#### Run resampling for pixel scale 

### PREP 

# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()

# set details for rasters
rasters_to_resample = {
    "economic": (economic, pixels_economic),
    "survival": (survival, pixels_survival),
}

### RESAMPLE
for name, (src_path, out_path) in rasters_to_resample.items():
    with rasterio.open(src_path) as src:
        src_nodata = src.nodata

        dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.average,
            src_nodata=src_nodata,
            dst_nodata=np.nan,
        )

    out_meta = dst_meta.copy()
    out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

    out_arr = dst_array.copy()
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Saved {name} → {out_path}")


Saved economic → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/economic_objective_prob.tif
Saved survival → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/survival_objective_prob.tif


In [4]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
with rasterio.open(pixels_economic) as src:
    raster_crs = src.crs

gdf_proj = total_geo.to_crs(raster_crs)

# set final form 
result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE
for col_name, rpath in [("probability_economic_land_use_objective", pixels_economic),
                         ("probability_survival_land_use_objective", pixels_survival)]:
    zonal = rasterstats.zonal_stats(
        gdf_proj,
        rpath,
        stats="mean",
        nodata=-9999
    )
    result[col_name] = [z["mean"] for z in zonal]

### SAVE
result.to_csv(vectors, index=False)